# Trace Count v16.2: Split-Local Indexed Shakespeare Windows

This version keeps the v16 task and four models (`RoPE/RPE x
non-thinking/thinking`) but changes the data loader. Tiny Shakespeare
is split before indexing; training windows are consumed without
replacement and natural count imbalance is preserved.


In [1]:
from pathlib import Path

IN_COLAB = False
DRIVE_READY = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
    DRIVE_READY = True
except ImportError:
    print("Local runtime: Google Drive mount skipped.")

DRIVE_RESULTS_ROOT = Path(
    "/content/drive/MyDrive/Colab_Notebooks/CoT_Counting/"
    "Synthetic_CoT_NiaH_Count/colab_results"
)
if DRIVE_READY:
    DRIVE_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print({"in_colab": IN_COLAB, "drive_ready": DRIVE_READY})


Mounted at /content/drive
{'in_colab': True, 'drive_ready': True}


## 1. Repository and environment


In [2]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Twist-Shan/Synthetic_CoT_NiaH_Count.git"
if IN_COLAB:
    ROOT = Path("/content/Synthetic_CoT_NiaH_Count")
    if not (ROOT / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    ROOT = next((path.resolve() for path in candidates if (path / "pyproject.toml").exists()), None)
    if ROOT is None:
        raise FileNotFoundError("Run this notebook from the repository or use Colab.")

os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print("repo =", ROOT)
print("python =", sys.executable)


repo = /content/Synthetic_CoT_NiaH_Count
python = /usr/bin/python3


In [3]:
def stream_command(command, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print(" ".join(map(str, command)), flush=True)
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        captured = []
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
            captured.append(line.rstrip())
        returncode = process.wait()
    if returncode:
        print("---- Last 160 log lines ----")
        print("\n".join(captured[-160:]))
        raise subprocess.CalledProcessError(returncode, command)


## 2. Runtime settings

The main preset uses 10,000 steps and four independent models. A
checkpoint is copied to Drive every configured checkpoint interval,
so reconnecting with `SKIP_COMPLETED=True` resumes both model and
sampler state exactly.


In [4]:
import torch
from synthetic_counting_v11.config import preset_config

PRESET = "main"  # "debug" first for a short end-to-end check
STAGE = "all"
SEED = 1234
MIN_CANDIDATE_WINDOWS = 128
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_ROOT = "runs/synthetic_counting_v16_2"
RUN_NAME = f"v16_2_{PRESET}_split_windows_seed{SEED}"
SKIP_COMPLETED = True
CHECKPOINT_SYNC_ROOT = DRIVE_RESULTS_ROOT / "v16_2_live_checkpoints" if DRIVE_READY else None

PLANNED_CONFIG = preset_config(
    "v16_2",
    PRESET,
    seed=SEED,
    device=DEVICE,
    min_candidate_windows=MIN_CANDIDATE_WINDOWS,
)
print({
    "run_name": RUN_NAME,
    "device": DEVICE,
    "models": PLANNED_CONFIG.model_variants,
    "split_fractions": (
        PLANNED_CONFIG.corpus_train_fraction,
        PLANNED_CONFIG.corpus_validation_fraction,
        1 - PLANNED_CONFIG.corpus_train_fraction - PLANNED_CONFIG.corpus_validation_fraction,
    ),
    "window_sampling": PLANNED_CONFIG.window_sampling,
    "min_candidate_windows": PLANNED_CONFIG.min_candidate_windows,
    "natural_class_imbalance": True,
})


ModuleNotFoundError: No module named 'synthetic_counting_v11'

## 3. Run training and analysis


In [ ]:
cmd = [
    sys.executable, "-u", "-m", "synthetic_counting_v16_2.run_v16_2",
    "--preset", PRESET,
    "--stage", STAGE,
    "--device", DEVICE,
    "--seed", str(SEED),
    "--min-candidate-windows", str(MIN_CANDIDATE_WINDOWS),
    "--out-root", OUT_ROOT,
    "--run-name", RUN_NAME,
]
if SKIP_COMPLETED:
    cmd.append("--skip-completed")
if CHECKPOINT_SYNC_ROOT is not None:
    cmd += ["--checkpoint-sync-root", str(CHECKPOINT_SYNC_ROOT)]
stream_command(cmd, Path(OUT_ROOT) / "last_pipeline.log")
RUN_DIR = Path(OUT_ROOT) / RUN_NAME
assert (RUN_DIR / "config.json").exists(), RUN_DIR
print("RUN_DIR =", RUN_DIR.resolve())


## 4. Inspect the indexed data distribution and results


In [ ]:
import pandas as pd
from IPython.display import Image, display

for filename in [
    "window_index_summary.csv",
    "training_count_distribution.csv",
    "final_accuracy_by_count.csv",
    "attention_head_summary.csv",
    "state_probe_summary.csv",
]:
    path = RUN_DIR / "tables" / filename
    if path.exists() and path.stat().st_size:
        print("\n", filename)
        display(pd.read_csv(path))

figures = sorted((RUN_DIR / "figures").glob("*.png"))
print(f"Generated {len(figures)} figures")
for path in figures:
    print(path.name)
    display(Image(filename=str(path)))


## 5. Save the complete result bundle to Google Drive


In [ ]:
import json
import shutil
from datetime import datetime

DRIVE_SAVE_COMPLETED = False
if DRIVE_READY:
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    destination = DRIVE_RESULTS_ROOT / f"{RUN_NAME}_{stamp}"
    if destination.exists():
        shutil.rmtree(destination)
    shutil.copytree(RUN_DIR, destination)
    DRIVE_SAVE_COMPLETED = True
    print("Saved result bundle:", destination)
else:
    print("Drive is unavailable; results remain at", RUN_DIR)


## 6. Optional GitHub step


In [ ]:
# Optional. Keep disabled unless you intentionally want to commit notebook/code changes.
PUSH_TO_GITHUB = False
if PUSH_TO_GITHUB:
    subprocess.run(["git", "status", "--short"], check=True)
    print("Review the status above, then commit and push intentionally from a terminal.")


## 7. Disconnect only after a verified Drive save


In [ ]:
AUTO_DISCONNECT_AFTER_DRIVE_SAVE = True
if IN_COLAB and AUTO_DISCONNECT_AFTER_DRIVE_SAVE and DRIVE_SAVE_COMPLETED:
    from google.colab import runtime
    print("Drive save verified; disconnecting the Colab runtime.")
    runtime.unassign()
else:
    print("Runtime kept alive (not Colab, save incomplete, or auto-disconnect disabled).")
